<a href="https://colab.research.google.com/drive/1s7DJchx8GA7kh1h7Z8zzCFp2XIHc1W66?usp=sharing" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Compiling Python with `numba`

Reproduce Python function from lecture and measure its execution time:

In [1]:
def loop(x, r):
    for i in range(r):
        x *= 2.5
    return x

%time loop(2, 10**6)

CPU times: total: 15.6 ms
Wall time: 29.2 ms


inf

## Using `numba`

First, let's try compiling "Just in Time" using `numba`:

In [2]:
from numba import jit

# jit compiles when we call the function for the first time
# nopython tries to run without involving Python interpreter
@jit(nopython=True)
def loop_jit(x, r):
  for i in range(r):
    x *= 2.5
  return x

%time loop_jit(2, 10**6) # includes compilation time

CPU times: total: 594 ms
Wall time: 900 ms


inf

In [3]:
%time loop_jit(2, 10**6) # much faster after compilation

CPU times: total: 0 ns
Wall time: 820 μs


inf

In [4]:
%timeit loop(2, 10**6) # better to time across multiple runs using `timeit`

26.7 ms ± 708 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [5]:
%timeit loop_jit(2, 10**6)

637 μs ± 20.8 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


We might want to compile our code ahead of time, though, so that we can see a speed-up the first time we use it. `numba` allows us to compile ahead of time like so:


In [6]:
from numba.pycc import CC

# name of compiled module to create:
cc = CC('test_aot')

# name of function in module, with explicit data types required (4byte=32bit ints and floats)
@cc.export('loop_aot', 'f4(f4,i4)')
def loop_aot(x, r):
    for i in range(r):
        x *= 2.5
    return x

cc.compile()

Note that we now have a compiled object file (.so) in our current directory. This is a compiled module that contains our function.

In [7]:
ls

 Volume in drive C is Windows
 Volume Serial Number is C6A9-BF22

 Directory of c:\Users\Tablet\Documents\GitHubRepositories\MACS 30113\course-materials\in-class-activities\01_Introduction

03/31/2026  11:08 PM    <DIR>          .
03/30/2026  03:26 PM    <DIR>          ..
03/30/2026  05:56 PM             7,590 1M_numba.ipynb
03/31/2026  11:08 PM            39,424 test_aot.cp313-win_amd64.pyd
               2 File(s)         47,014 bytes
               2 Dir(s)  763,512,401,920 bytes free


To use our function, we just need to import our pre-compiled module, as we would any other Python module:

In [8]:
import test_aot
%time test_aot.loop_aot(2, 10**6) # first time running it is fast this time

CPU times: total: 0 ns
Wall time: 770 μs


inf

In [9]:
%timeit test_aot.loop_aot(2, 10**6) # same overall performance as before

729 μs ± 16 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
